In [1]:
!pip install torch torchvision pillow

   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 1.0/124.1 MB 7.1 MB/s eta 0:00:18
    --------------------------------------- 2.1/124.1 MB 4.9 MB/s eta 0:00:25
    --------------------------------------- 2.1/124.1 MB 4.9 MB/s eta 0:00:25
    --------------------------------------- 2.1/124.1 MB 4.9 MB/s eta 0:00:25
    --------------------------------------- 2.6/124.1 MB 2.8 MB/s eta 0:00:44
    --------------------------------------- 2.6/124.1 MB 2.8 MB/s eta 0:00:44
    --------------------------------------- 2.9/124.1 MB 1.8 MB/s eta 0:01:06
    --------------------------------------- 2.9/124.1 MB 1.8 MB/s eta 0:01:06
   - -------------------------------------- 3.1/124.1 MB 1.6 MB/s eta 0:01:14
   - -------------------------------------- 3.1/124.1 MB 1.6 MB/s eta 0:01:14
   - -------------------------------------- 3.1/124.1 MB 1.6 MB/s eta 0:01:14
   - -------------------------------------- 3.4/124.1 MB 1.3 MB/s eta 0

In [2]:
from PIL import Image
from torchvision import transforms

def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return transform(image).unsqueeze(0)

In [5]:
import torch
from torchvision import models

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.eval()

input_image = preprocess_image("tiger.jpg")   

with torch.no_grad():
    output = model(input_image)
    predicted_class = output.argmax(1).item()

class_names = weights.meta["categories"]     
print("Predicted Class Index:", predicted_class)
print("Predicted Class Label:", class_names[predicted_class])

Predicted Class Index: 292
Predicted Class Label: tiger


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))   # Fixed: 3 channels
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

net = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)


for epoch in range(5):
    net.train()
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader):.4f}")


net.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

100.0%


Epoch 1, Loss: 1.3429
Epoch 2, Loss: 0.9664
Epoch 3, Loss: 0.8160
Epoch 4, Loss: 0.7020
Epoch 5, Loss: 0.6000
Test Accuracy: 71.75%


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = to_categorical(y_train), to_categorical(y_test)

model = Sequential([
    Input(shape=(28, 28)),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model.fit(x_train, y_train, validation_data=(x_test, y_test),
          epochs=30, batch_size=128, callbacks=[early_stop])

loss, acc = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {acc:.4f}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Epoch 1/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.8870 - loss: 0.3795 - val_accuracy: 0.9578 - val_loss: 0.1380
Epoch 2/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9511 - loss: 0.1670 - val_accuracy: 0.9693 - val_loss: 0.1055
Epoch 3/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9629 - loss: 0.1249 - val_accuracy: 0.9725 - val_loss: 0.0901
Epoch 4/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9691 - loss: 0.1012 - val_accuracy: 0.9760 - val_loss: 0.0759
Epoch 5/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9735 - loss: 0.0869 - val_accuracy: 0.9779 - val_loss: 0.0725
Epoch 6/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9774 - loss: 0.0746 - val_accuracy: 0.9801 - val_loss: 0.0683
Epoch 7/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9781 - loss: 0.0700 - val_accuracy: 0.9772 - val_loss: 0.0726
Epoch 8/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step -

In [7]:
pip install tensorflow-datasets

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ------- -------------------------------- 1.0/5.3 MB 7.1 MB/s eta 0:00:01
   --------- ------------------------------ 1.3/5.3 MB 3.5 MB/s eta 0:00:02
   --------------------- ------------------ 2.9/5.3 MB 4.5 MB/s eta 0:00:01
   --------------------------- ------------ 3.7/5.3 MB 4.4 MB/s eta 0:00:01
   --------------------------------- ------ 4.5/5.3 MB 4.3 MB/s eta 0:00:01
   ---------------------------------------  5.2/5.3 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 5.3/5.3 MB 4.0 MB/s  0:00:01
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
  

In [10]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.applications.vgg16 import preprocess_input

train_raw, val_raw = tfds.load(
    'cats_vs_dogs',
    split=['train[:20%]', 'train[20%:25%]'],
    as_supervised=True
)

def prep(image, label):
    image = tf.image.resize(image, (224, 224))
    image = preprocess_input(image)
    label = tf.cast(label, tf.float32)
    return image, tf.reshape(label, [1])

train_ds = train_raw.map(prep).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)
val_ds = val_raw.map(prep).batch(32).prefetch(tf.data.AUTOTUNE)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

KeyError: "There is no item named 'PetImages\\\\Cat\\\\0.jpg' in the archive"

In [9]:
!pip install importlib_resources

In [11]:
!pip install tensorflow pillow

In [14]:
zip_file = "cats_dogs_light.zip"
extract_dir = "cats_dogs_data"

if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_file, "r") as z:
        z.extractall(extract_dir)

# Find folders that contain images and group them by parent folder
splits = {}
for root, dirs, files in os.walk(extract_dir):
    if "__MACOSX" in root:
        continue
    imgs = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if imgs:
        parent = os.path.dirname(root)
        splits[parent] = splits.get(parent, 0) + len(imgs)

print("Found image folders:", splits)

# Bigger one = train (1000 images), smaller one = validation (400 images)
ordered = sorted(splits, key=splits.get, reverse=True)
train_dir, val_dir = ordered[0], ordered[1]
print("Train:", train_dir)
print("Validation:", val_dir)

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=(224, 224), batch_size=32, label_mode="binary")
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=(224, 224), batch_size=32, label_mode="binary")

train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))

ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-153' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Anaconda\envs\aiml\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-154' coro=<Kernel.shell_main() running at C:\Anaconda\envs\aiml\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Anaconda\envs\aiml\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Anaconda\envs\aiml\Lib\zipfile.py:1433: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  filename = filename.decode(self.metadata_encoding or 'cp437')
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-154' coro=<Kernel.shell_main() running at C:\Anaconda\envs\aiml\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>


Found image folders: {'cats_dogs_data\\cats_dogs_light': 1389}


IndexError: list index out of range

In [15]:
import os
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input

base = os.path.join("cats_dogs_data", "cats_dogs_light")
IMG_EXT = (".jpg", ".jpeg", ".png")

subfolders = [d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))]
print("Folders inside cats_dogs_light:")
for d in subfolders:
    p = os.path.join(base, d)
    print("  ", d, "->", len(os.listdir(p)), "items, e.g.", os.listdir(p)[:3])

names = [d.lower() for d in subfolders]
looks_like_classes = all(("cat" in n or "dog" in n) for n in names)

if looks_like_classes:
    # CASE A: folders are the classes (cats / dogs) -> split 80/20 automatically
    print("\nLayout: class folders, using an 80/20 train/validation split")
    train_ds = tf.keras.utils.image_dataset_from_directory(
        base, validation_split=0.2, subset="training", seed=123,
        image_size=(224, 224), batch_size=32, label_mode="binary")
    val_ds = tf.keras.utils.image_dataset_from_directory(
        base, validation_split=0.2, subset="validation", seed=123,
        image_size=(224, 224), batch_size=32, label_mode="binary")
    train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
    val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))

else:
    # CASE B: folders are train/test, labels come from file names (cat.1.jpg, dog.5.jpg)
    print("\nLayout: split folders, labels taken from file names")
    train_folder = [d for d in subfolders if "train" in d.lower()][0]
    val_folder = [d for d in subfolders if d != train_folder][0]

    def make_ds(folder, shuffle):
        folder_path = os.path.join(base, folder)
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(IMG_EXT)]
        paths = [os.path.join(folder_path, f) for f in files]
        labels = [0.0 if "cat" in f.lower() else 1.0 for f in files]   # cat=0, dog=1
        print(folder, "->", len(paths), "images,", int(sum(labels)), "dogs")

        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if shuffle:
            ds = ds.shuffle(len(paths), seed=123)

        def load(path, label):
            img = tf.io.read_file(path)
            img = tf.io.decode_image(img, channels=3, expand_animations=False)
            img = tf.image.resize(img, (224, 224))
            img = preprocess_input(img)
            return img, tf.reshape(label, [1])

        return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

    train_ds = make_ds(train_folder, shuffle=True)
    val_ds = make_ds(val_folder, shuffle=False)

Folders inside cats_dogs_light:
   test -> 400 items, e.g. ['cat.9818.jpg', 'cat.9819.jpg', 'cat.9820.jpg']
   train -> 989 items, e.g. ['cat.0.jpg', 'cat.1.jpg', 'cat.10.jpg']

Layout: split folders, labels taken from file names
train -> 989 images, 534 dogs
test -> 400 images, 200 dogs


In [16]:
base_model = VGG16(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = Flatten()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 17s 0us/step


In [17]:
model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5


C:\Anaconda\envs\aiml\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


31/31 ━━━━━━━━━━━━━━━━━━━━ 81s 3s/step - accuracy: 0.8726 - loss: 4.3085 - val_accuracy: 0.9225 - val_loss: 0.9384
Epoch 2/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 85s 3s/step - accuracy: 0.9778 - loss: 0.2705 - val_accuracy: 0.9800 - val_loss: 0.1239
Epoch 3/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 92s 3s/step - accuracy: 0.9909 - loss: 0.0664 - val_accuracy: 0.9900 - val_loss: 0.0562
Epoch 4/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 99s 3s/step - accuracy: 0.9980 - loss: 0.0169 - val_accuracy: 0.9875 - val_loss: 0.0567
Epoch 5/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 105s 3s/step - accuracy: 1.0000 - loss: 0.0016 - val_accuracy: 0.9925 - val_loss: 0.0498


In [18]:
for layer in base_model.layers:
    layer.trainable = layer.name.startswith("block5")

model.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 92s 3s/step - accuracy: 1.0000 - loss: 7.7317e-04 - val_accuracy: 0.9925 - val_loss: 0.0451
Epoch 2/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 100s 3s/step - accuracy: 1.0000 - loss: 9.0682e-04 - val_accuracy: 0.9925 - val_loss: 0.0511
Epoch 3/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 110s 4s/step - accuracy: 0.9990 - loss: 0.0027 - val_accuracy: 0.9925 - val_loss: 0.0514
Epoch 4/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 117s 4s/step - accuracy: 0.9980 - loss: 0.0055 - val_accuracy: 0.9900 - val_loss: 0.0766
Epoch 5/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 118s 4s/step - accuracy: 1.0000 - loss: 1.5610e-04 - val_accuracy: 0.9925 - val_loss: 0.0422
